# Feature Engineering

Feature Engineering приносит большую отдачу, чем построение модели и настройка гиперпараметров.

Хотя выбор правильной модели и оптимальных настроек важен, модель может учиться только на предоставленных ей данных.

### Импорт библиотек и вспомогательные функции

In [ ]:
import pandas as pd
import numpy as np
import os
import gc
from sklearn.preprocessing import LabelEncoder

RAW_DATA_DIR = '../data/raw/'
PROCESSED_DATA_DIR = '../data/processed/'


def one_hot_encoder(df, nan_as_category=True):
    '''Вспомогательная функция для One-Hot Encoding категориальных признаков'''
    original_columns = list(df.columns)
    categorical_columns = [col for col in df.columns if df[col].dtype == 'object']
    df = pd.get_dummies(df, columns=categorical_columns, dummy_na=nan_as_category)
    new_columns = [c for c in df.columns if c not in original_columns]
    return df, new_columns

### 1. Обработка application_train & application_test

In [ ]:
def process_application():
    df_train = pd.read_csv(os.path.join(RAW_DATA_DIR, 'application_train.csv'))
    df_test = pd.read_csv(os.path.join(RAW_DATA_DIR, 'application_test.csv'))
    df = pd.concat([df_train, df_test], axis=0, ignore_index=True)
    del df_train, df_test; gc.collect()

    # Обработка аномалий
    df['DAYS_EMPLOYED'].replace(365243, np.nan, inplace=True)
    df['DAYS_EMPLOYED_ANOM'] = df["DAYS_EMPLOYED"].isnull().astype(int) # Флаг аномалии стажа (1 - аномальный стаж, 0 - реальный стаж)
    df['DAYS_LAST_PHONE_CHANGE'].replace(0, np.nan, inplace=True)

    # Финансовые индикаторы
    df['CREDIT_INCOME_PERCENT'] = df['AMT_CREDIT'] / df['AMT_INCOME_TOTAL'] # Сколько годовых доходов составляет запрашиваемый кредит
    df['ANNUITY_INCOME_PERCENT'] = df['AMT_ANNUITY'] / df['AMT_INCOME_TOTAL'] # Отношение ежегодного платежа к годовому доходу
    df['CREDIT_TERM'] = df['AMT_ANNUITY'] / df['AMT_CREDIT'] # "Скорость" выплаты долга
    df['DAYS_EMPLOYED_PERCENT'] = df['DAYS_EMPLOYED'] / df['DAYS_BIRTH'] # Отношение стажа к возрасту
    df['CREDIT_GOODS_RATIO'] = df['AMT_CREDIT'] / df['AMT_GOODS_PRICE'] # Отношение суммы кредита к цене товара
    df['CREDIT_DOWNPAYMENT'] = df['AMT_GOODS_PRICE'] - df['AMT_CREDIT'] # Сумма первоначального взноса.
    # Разница между ценой товара и выданным кредитом
    df['AGE_INT'] = (df['DAYS_BIRTH'] / -365).astype(int) # Возраст клиента в полных годах
    
    # Продвинутые признаки
    df['INCOME_TO_EXT3'] = df['AMT_INCOME_TOTAL'] / (df['EXT_SOURCE_3'] + 0.001) # Отношение дохода к баллу из внешнего источника №3.
    # Попытка взвесить доход клиента с учетом его надежности по данным сторонних организаций
    df['EXT_SOURCES_MEAN'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1) # Обобщенная оценка надежности клиента исходя из внешних
    df['EXT_SOURCES_PROD'] = df['EXT_SOURCE_1'] * df['EXT_SOURCE_2'] * df['EXT_SOURCE_3'] # Этот показатель «штрафует» клиента, если хотя бы один из показателей очень низкий

    # Категориальные признаки
    # Label Encoding для бинарных признаков
    le = LabelEncoder()
    for col in df.columns:
        if df[col].dtype == 'object':
            if len(list(df[col].unique())) <= 2:
                le.fit(df[col])
                df[col] = le.transform(df[col])
    
    # One-Hot Encoding для остальных
    df, _ = one_hot_encoder(df)
    
    # REGION_ID_POPULATION как категория (совет Phil)
    df['REGION_ID_POPULATION'] = df['REGION_ID_POPULATION'].astype('category')

    return df

df = process_application()
print(f"Application shape: {df.shape}")

### 2. Bureau & Bureau Balance

In [ ]:
def process_bureau():
    bureau = pd.read_csv(os.path.join(RAW_DATA_DIR, 'bureau.csv'))
    bb = pd.read_csv(os.path.join(RAW_DATA_DIR, 'bureau_balance.csv'))
    
    # Bureau Balance: Агрегация статусов
    bb, bb_cat = one_hot_encoder(bb)
    bb_aggregations = {'MONTHS_BALANCE': ['min', 'max', 'size']}
    for col in bb_cat:
        bb_aggregations[col] = ['mean']
    bb_agg = bb.groupby('SK_ID_BUREAU').agg(bb_aggregations)
    bb_agg.columns = pd.Index([e[0] + "_" + e[1].upper() for e in bb_agg.columns.tolist()])
    bureau = bureau.join(bb_agg, how='left', on='SK_ID_BUREAU')
    bureau.drop(['SK_ID_BUREAU'], axis=1, inplace=True)
    del bb, bb_agg; gc.collect()

    # Bureau: Числовые агрегации
    bureau, bureau_cat = one_hot_encoder(bureau)
    
    bureau['BUREAU_DEBT_CREDIT_RATIO'] = bureau['AMT_CREDIT_SUM_DEBT'] / bureau['AMT_CREDIT_SUM']
    
    num_aggregations = {
        'DAYS_CREDIT': ['mean', 'max', 'min', 'var'],
        'DAYS_CREDIT_ENDDATE': ['mean', 'max'],
        'AMT_CREDIT_MAX_OVERDUE': ['mean', 'max'],
        'AMT_CREDIT_SUM': ['mean', 'max', 'sum'],
        'AMT_CREDIT_SUM_DEBT': ['mean', 'max', 'sum'],
        'BUREAU_DEBT_CREDIT_RATIO': ['mean', 'max']
    }
    
    cat_aggregations = {}
    for col in bureau_cat: cat_aggregations[col] = ['mean']
    
    bureau_agg = bureau.groupby('SK_ID_CURR').agg({**num_aggregations, **cat_aggregations})
    bureau_agg.columns = pd.Index(['BURO_' + e[0] + "_" + e[1].upper() for e in bureau_agg.columns.tolist()])
    
    # Самое последнее DAYS_CREDIT для активных кредитов
    active = bureau[bureau['CREDIT_ACTIVE_Active'] == 1]
    active_agg = active.groupby('SK_ID_CURR').agg({'DAYS_CREDIT': ['max']})
    active_agg.columns = pd.Index(['BURO_LAST_ACTIVE_DAYS_CREDIT_MAX'])
    bureau_agg = bureau_agg.join(active_agg, how='left', on='SK_ID_CURR')
    
    del bureau; gc.collect()
    return bureau_agg

bureau_agg = process_bureau()
df = df.join(bureau_agg, how='left', on='SK_ID_CURR')
print(f"Bureau added. Shape: {df.shape}")

### 3. Previous Applications
Прошлые заявки

In [ ]:
def process_previous():
    prev = pd.read_csv(os.path.join(RAW_DATA_DIR, 'previous_application.csv'))
    prev, cat_cols = one_hot_encoder(prev)
    
    # Исправление дат
    prev['DAYS_FIRST_DRAWING'].replace(365243, np.nan, inplace=True)
    prev['DAYS_FIRST_DUE'].replace(365243, np.nan, inplace=True)
    prev['DAYS_LAST_DUE_1ST_VERSION'].replace(365243, np.nan, inplace=True)
    prev['DAYS_LAST_DUE'].replace(365243, np.nan, inplace=True)
    prev['DAYS_TERMINATION'].replace(365243, np.nan, inplace=True)

    prev['ASKED_AMT_ID_CURR_DIFF'] = prev['AMT_APPLICATION'] - prev['AMT_CREDIT'] # Разница между суммой, которую клиент просил, и суммой, которую банк одобрил
    prev['APPLICATION_CREDIT_RATIO'] = prev['AMT_APPLICATION'] / prev['AMT_CREDIT'] # Отношение запрошенной суммы к одобренной.
    # Показывает степень доверия банка к клиенту в прошлых сделках
    
    # Агрегации
    num_aggregations = {
        'AMT_ANNUITY': ['mean', 'max'],
        'AMT_APPLICATION': ['mean', 'max'],
        'AMT_CREDIT': ['mean', 'max'],
        'AMT_DOWN_PAYMENT': ['mean', 'max'],
        'AMT_GOODS_PRICE': ['mean', 'max'],
        'HOUR_APPR_PROCESS_START': ['mean', 'max'],
        'RATE_DOWN_PAYMENT': ['mean', 'max'],
        'DAYS_DECISION': ['mean', 'max'],
        'CNT_PAYMENT': ['mean', 'sum'],
    }
    cat_aggregations = {}
    for col in cat_cols: cat_aggregations[col] = ['mean']
    
    prev_agg = prev.groupby('SK_ID_CURR').agg({**num_aggregations, **cat_aggregations})
    prev_agg.columns = pd.Index(['PREV_' + e[0] + "_" + e[1].upper() for e in prev_agg.columns.tolist()])
    
    # Последняя заявка
    last_app = prev.sort_values(by='DAYS_DECISION', ascending=False).groupby('SK_ID_CURR').first()
    last_app_cols = [c for c in last_app.columns if 'NAME_CONTRACT_STATUS' in c or 'PRODUCT_COMBINATION' in c]
    last_app = last_app[last_app_cols]
    last_app.columns = ['LAST_APP_' + c for c in last_app.columns]
    
    prev_agg = prev_agg.join(last_app, how='left', on='SK_ID_CURR')
    
    del prev; gc.collect()
    return prev_agg

prev_agg = process_previous()
df = df.join(prev_agg, how='left', on='SK_ID_CURR')
print(f"Previous apps added. Shape: {df.shape}")

### 4. Installments Payments

In [ ]:
def process_installments():
    ins = pd.read_csv(os.path.join(RAW_DATA_DIR, 'installments_payments.csv'))
    
    ins['PAYMENT_PERC'] = ins['AMT_PAYMENT'] / ins['AMT_INSTALMENT']
    ins['PAYMENT_DIFF'] = ins['AMT_INSTALMENT'] - ins['AMT_PAYMENT']
    
    # Просрочка
    ins['DPD'] = ins['DAYS_ENTRY_PAYMENT'] - ins['DAYS_INSTALMENT']
    ins['DPD'] = ins['DPD'].apply(lambda x: x if x > 0 else 0)
    ins['DBD'] = ins['DAYS_INSTALMENT'] - ins['DAYS_ENTRY_PAYMENT']
    ins['DBD'] = ins['DBD'].apply(lambda x: x if x > 0 else 0)

    # Агрегация по SK_ID_CURR
    aggregations = {
        'NUM_INSTALMENT_VERSION': ['nunique'],
        'DPD': ['max', 'mean', 'sum'],
        'DBD': ['max', 'mean', 'sum'],
        'PAYMENT_PERC': ['mean', 'max', 'var'],
        'PAYMENT_DIFF': ['mean', 'max', 'sum', 'var'],
        'AMT_INSTALMENT': ['max', 'mean', 'sum'],
        'AMT_PAYMENT': ['min', 'max', 'mean', 'sum'],
        'DAYS_ENTRY_PAYMENT': ['max', 'mean', 'sum']
    }
    
    ins_agg = ins.groupby('SK_ID_CURR').agg(aggregations)
    ins_agg.columns = pd.Index(['INS_' + e[0] + "_" + e[1].upper() for e in ins_agg.columns.tolist()])
    ins_agg['INS_COUNT'] = ins.groupby('SK_ID_CURR').size()

    # Временное окно: последние 1000 дней
    ins_1000 = ins[ins['DAYS_INSTALMENT'] > -1000]
    ins_1000_agg = ins_1000.groupby('SK_ID_CURR').agg({'PAYMENT_DIFF': ['mean']})
    ins_1000_agg.columns = ['INS_1000_PAYMENT_DIFF_MEAN']
    ins_agg = ins_agg.join(ins_1000_agg, how='left', on='SK_ID_CURR')

    # Временные окна: последние 365 дней
    ins_365 = ins[ins['DAYS_INSTALMENT'] > -365]
    ins_365_agg = ins_365.groupby('SK_ID_CURR').agg({'DPD': ['sum'], 'PAYMENT_DIFF': ['mean']})
    ins_365_agg.columns = ['INS_365_DPD_SUM', 'INS_365_PAYMENT_DIFF_MEAN']
    ins_agg = ins_agg.join(ins_365_agg, how='left', on='SK_ID_CURR')

    # Это будет рассчитано после объединения с основной таблицей
    del ins; gc.collect()
    return ins_agg

ins_agg = process_installments()
df = df.join(ins_agg, how='left', on='SK_ID_CURR')

df['ANNUITY_TO_MAX_INS_RATIO'] = df['AMT_ANNUITY'] / (df['INS_AMT_INSTALMENT_MAX'] + 0.001)

print(f"Installments added. Shape: {df.shape}")

### 5. POS_CASH & Credit Card Balance

Агрегации и One-Hot Encoding (из EDA).

In [ ]:
def process_pos_cash():
    pos = pd.read_csv(os.path.join(RAW_DATA_DIR, 'POS_CASH_balance.csv'))
    pos, cat_cols = one_hot_encoder(pos)
    
    aggregations = {
        'MONTHS_BALANCE': ['max', 'mean', 'size'],
        'SK_DPD': ['max', 'mean'],
        'SK_DPD_DEF': ['max', 'mean']
    }
    for col in cat_cols: aggregations[col] = ['mean']
    
    pos_agg = pos.groupby('SK_ID_CURR').agg(aggregations)
    pos_agg.columns = pd.Index(['POS_' + e[0] + "_" + e[1].upper() for e in pos_agg.columns.tolist()])
    
    # Кол-во активных кредитов
    active_counts = pos[pos['NAME_CONTRACT_STATUS_Active'] == 1].groupby('SK_ID_CURR').size()
    pos_agg['POS_ACTIVE_COUNT'] = active_counts
    
    del pos; gc.collect()
    return pos_agg

def process_credit_card():
    cc = pd.read_csv(os.path.join(RAW_DATA_DIR, 'credit_card_balance.csv'))
    cc, cat_cols = one_hot_encoder(cc)
    
    # Заполняем нулями поля задолженностей
    cc.drop(['SK_ID_PREV'], axis=1, inplace=True)
    cc_agg = cc.groupby('SK_ID_CURR').agg(['min', 'max', 'mean', 'sum', 'var'])
    cc_agg.columns = pd.Index(['CC_' + e[0] + "_" + e[1].upper() for e in cc_agg.columns.tolist()])
    
    # Признак использование лимита
    cc_agg['CC_LIMIT_USE_MEAN'] = cc_agg['CC_AMT_BALANCE_MEAN'] / (cc_agg['CC_AMT_CREDIT_LIMIT_ACTUAL_MEAN'] + 0.001)
    
    del cc; gc.collect()
    return cc_agg

pos_agg = process_pos_cash()
df = df.join(pos_agg, how='left', on='SK_ID_CURR')

cc_agg = process_credit_card()
df = df.join(cc_agg, how='left', on='SK_ID_CURR')

print(f"Final Data shape: {df.shape}")

### 6. Финальное объединение и очистка

In [ ]:
# Удаление столбцов с нулевой вариативностью
for col in df.columns:
    if df[col].nunique() <= 1:
        df.drop(col, axis=1, inplace=True)

df_train = df[df['TARGET'].notnull()]
df_test = df[df['TARGET'].isnull()]

print(f"Train shape: {df_train.shape}")
print(f"Test shape: {df_test.shape}")

# Сохранение результатов
# df_train.to_csv('../data/processed/train_features.csv', index=False)
# df_test.to_csv('../data/processed/test_features.csv', index=False)

### **Основные внесенные изменения:**

- Добавлены соотношения CREDIT_GOODS_RATIO, CREDIT_DOWNPAYMENT, ANNUITY_TO_MAX_INS_RATIO и учет REGION_ID_POPULATION как категории
- Реализованы лаговые признаки из последних заявок и агрегации платежей за последние 365 дней. Деление дохода на внешние источники
- Продвинутые агрегаты по AMT_ANNUITY и CNT_PAYMENT в предыдущих заявках для оценки "стоимости" кредита
- Созданы KPI на основе использования кредитного лимита и частоты активных POS-кредитов
- из EDA: Обработаны аномалии в DAYS_EMPLOYED, созданы флаги пропусков и применен One-Hot Encoding для всех динамических таблиц перед агрегацией.